In [7]:
#!pip install transformers --break-system-packages
#!pip install "transformers[torch]"

Access is denied.


In [1]:
import pandas as pd 
from transformers import T5Tokenizer , Trainer ,TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("samsun_train.csv",sep ='\t', on_bad_lines='skip')
val_data = pd.read_csv("samsun_validation.csv",sep='\t' ,on_bad_lines='skip')
train_data
val_data

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrowâ€™s afternoon...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: Iâ€™ve just fallen in love with this adv...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\nJackie: but she d...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\nMarla: look what I found ...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...
...,...,...,...
813,13829423,Carla: I've got it...\nDiego: what?\nCarla: my...,Carla's date for graduation is on June 4th. Di...
814,13727710,"Gita: Hello, this is Beti's Mum Gita, I wanted...",Bev is going on the school trip with her son. ...
815,13829261,"Julia: Greg just texted me\nRobert: ugh, delet...",Greg cheated on Julia. He apologises to her. R...
816,13680226,"Marry: I broke my nail ;(\nTina: oh, no!\nMarr...",Marry broke her nail and has a party tomorrow....


In [3]:
#random sampling 

train_data= train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data= val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [4]:
train_data.shape

(4000, 3)

pre processing


In [5]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",text) # remove lines 
    text = re.sub(r"\s+"," ",text) #remove space
    text = re.sub(r"<.*>?>"," ", text) # remove html tags
    text = text.strip().lower()
    return text

In [6]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = train_data["dialogue"].apply(clean_data)
val_data["summary"] = train_data["summary"].apply(clean_data)

In [7]:
train_data

,id,dialogue,summary
0,13811908,violet: hi! i came across this austin's articl...,violet sent claire austin's article.
1,13716431,pat: so does anyone know when the stream is go...,pat and lou are waiting for the stream but kev...
2,13810214,jane: jane: whaddya think? shona: this ur ti...,jane is updating her tinder profile tonight an...
3,13729823,"adam: do u have a map of paris? tom: yes, why?...",tom has a map of paris.
4,13681400,"frank: hi, how's the family? mike: great! sam'...","mike is happy, because sam's moved out. mike a..."
...,...,...,...
3995,13681041,barry: hello buddy michael: hey barry: do you ...,barry and michael will watch football instead ...
3996,13818705,karen: hey lisa. larissa and me have recently ...,karen and larissa moved to belgium and ask lis...
3997,13821859,"miles: hey, guys, i'm so sorry, but i missed t...","miles has missed the bus, so he may be 15 minu..."
3998,13812716,emma: did you finish the book i gave you? liam...,"emma gave ""the first fifteen lives of harry au..."


Tokenize

In [8]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [9]:
# raw data => tokenized inputs for fine tuning 
def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding = "max_length",max_length=512,truncation = True)
    target = tokenizer(data["summary"],padding = "max_length",max_length=150,truncation = True)
 
    inputs["labels"] = target["input_ids"]
    return inputs
   
    

In [10]:
train_dataset= train_data.apply(tokenize,axis =1).tolist()
val_dataset= val_data.apply(tokenize,axis =1).tolist()

In [11]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [12]:
# NLP generation Task
model = T5ForConditionalGeneration.from_pretrained("t5-small")



Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [13]:
import torch
# device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [14]:

# import sys
# import torch
# import transformers
# import accelerate

# print("Python:", sys.executable)
# print("Torch:", torch.__version__)
# print("Transformers:", transformers.__version__)
# print("Accelerate:", accelerate.__version__)
# import sys
# print(sys.executable)
# import sys

# !{sys.executable} -m pip install accelerate

#training arguments
# training_args =  TrainingArguments(
#     output_dir = "./results",
#     num_train_epochs = 6,
#     weight_decay = 0.01,
#     per_device_train_batch_size =8,
#     per_device_eval_batch_size =8,
#     eval_strategy = "epoch",
#     save_strategy = "epoch",
#     warmup_steps = 500
    
# )
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500,
    use_cpu=True
)

print("TrainingArguments created successfully")

TrainingArguments created successfully


In [15]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Trainer created successfully")

Trainer created successfully


In [ ]:
#train the model
trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
#model load => fine tune => save the model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer("./saved_summary_model")


In [ ]:
#test core logic for summarization
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)
    #tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    #generate summary => token_ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping = True
    )
    #decoded our output
    summary = tokenizer.decode(targets[0],skip_special_tokens=True)
    return summary
    
    
    